In [2]:
:clear

In [ ]:
:dep pecos = { version = "0.1.1", path = "../crates/pecos" }

In [ ]:
// Bell State Example using SparseStab simulator
use pecos::prelude::*;

// Create a 2-qubit stabilizer simulator
let mut sim = StdSparseStab::new(2);

// Create Bell state: (|00⟩ + |11⟩)/√2
sim.h(0)      // Apply Hadamard to qubit 0: |00⟩ -> (|00⟩ + |10⟩)/√2
   .cx(0, 1); // Apply CNOT: (|00⟩ + |10⟩)/√2 -> (|00⟩ + |11⟩)/√2

// Measure both qubits
let result0: MeasurementResult = sim.mz(0);
let result1: MeasurementResult = sim.mz(1);

println!("Bell State Measurement Results:");
println!("  Qubit 0: {} (deterministic: {})", 
         if result0.outcome { "1" } else { "0" },
         result0.is_deterministic);
println!("  Qubit 1: {} (deterministic: {})", 
         if result1.outcome { "1" } else { "0" },
         result1.is_deterministic);

// Verify Bell state correlation: both qubits should always measure the same
assert_eq!(result0.outcome, result1.outcome, 
           "Bell state measurements must be correlated!");
println!("\nSuccess! Measurements are correlated as expected for a Bell state.");

In [ ]:
// Symbolic Bell State Example using SymbolicSparseStab
// This simulator tracks measurement dependencies instead of collapsing to concrete outcomes

let mut sym_sim = StdSymbolicSparseStab::new(2);

// Create Bell state: (|00⟩ + |11⟩)/√2
sym_sim.h(0).cx(0, 1);

// Measure both qubits
let r0: SymbolicMeasurementResult = sym_sim.mz(0);
let r1: SymbolicMeasurementResult = sym_sim.mz(1);

println!("Symbolic Bell State Measurement Results:");
println!("  Measurement 0: {:?} ^ {} (deterministic: {})", r0.outcome, r0.flip as u8, r0.is_deterministic);
println!("  Measurement 1: {:?} ^ {} (deterministic: {})", r1.outcome, r1.flip as u8, r1.is_deterministic);

// The first measurement is non-deterministic (creates measurement index 0)
// The second measurement is deterministic and depends on measurement 0
println!("\nAnalysis:");
if !r0.is_deterministic {
    println!("  - Qubit 0 measurement was non-deterministic (random outcome)");
}
if r1.is_deterministic && r0.outcome == r1.outcome {
    println!("  - Qubit 1 measurement is deterministic and depends on measurement 0");
    println!("  - This shows the Bell state correlation: both qubits always measure the same!");
}

In [ ]:
// 3-qubit Repetition Code in Logical |+_L⟩ with Syndrome Measurements
//
// Qubits:
//   q0, q1, q2: Data qubits 
//   q3: Ancilla for Z0Z1 check
//   q4: Ancilla for Z1Z2 check
//
// Logical states:
//   |0_L⟩ = |000⟩
//   |1_L⟩ = |111⟩
//   |+_L⟩ = (|000⟩ + |111⟩)/√2
//
// Encoding circuit for |+_L⟩:
//   - Start with |+⟩|00⟩ (H on q0)
//   - CX(0,1), CX(0,2) to spread → (|000⟩ + |111⟩)/√2
//
// Syndrome extraction:
//   - Measure Z0Z1 using ancilla q3
//   - Measure Z1Z2 using ancilla q4
//
// Then measure data qubits in Z basis

let mut sim = StdSymbolicSparseStab::new(5);

println!("=== 3-Qubit Repetition Code: Logical |+_L⟩ ===\n");

// Encode logical |+_L⟩ = (|000⟩ + |111⟩)/√2
sim.h(0);           // |+⟩|00⟩
sim.cx(0, 1);       // (|00⟩ + |11⟩)|0⟩
sim.cx(0, 2);       // |000⟩ + |111⟩
println!("After encoding |+_L⟩ = (|000⟩ + |111⟩)/√2:");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Syndrome measurement: Z0Z1 via ancilla q3
// Circuit: H(q3), CX(q0,q3), CX(q1,q3), H(q3), Measure(q3)
sim.h(3);
sim.cx(0, 3);
sim.cx(1, 3);
sim.h(3);
println!("After Z0Z1 syndrome circuit (before measurement):");
println!("Stabilizers:\n{}", sim.stab_tableau());

let s0: SymbolicMeasurementResult = sim.mz(3);
println!("Syndrome S0 (Z0Z1) = {:?} ^ {}, det={}", s0.outcome, s0.flip as u8, s0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

// Syndrome measurement: Z1Z2 via ancilla q4
sim.h(4);
sim.cx(1, 4);
sim.cx(2, 4);
sim.h(4);
println!("After Z1Z2 syndrome circuit (before measurement):");
println!("Stabilizers:\n{}", sim.stab_tableau());

let s1: SymbolicMeasurementResult = sim.mz(4);
println!("Syndrome S1 (Z1Z2) = {:?} ^ {}, det={}", s1.outcome, s1.flip as u8, s1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure data qubits in Z basis
let d0: SymbolicMeasurementResult = sim.mz(0);
println!("D0 (q0) = {:?} ^ {}, det={}", d0.outcome, d0.flip as u8, d0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let d1: SymbolicMeasurementResult = sim.mz(1);
println!("D1 (q1) = {:?} ^ {}, det={}", d1.outcome, d1.flip as u8, d1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let d2: SymbolicMeasurementResult = sim.mz(2);
println!("D2 (q2) = {:?} ^ {}, det={}", d2.outcome, d2.flip as u8, d2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

println!("\n=== Summary ===");
println!("Measurement indices: S0=0, S1=1, D0=2, D1=3, D2=4");
println!("\nSyndromes should be deterministic {{}} (no errors in code space).");
println!("Data qubits: D0 is random, D1 and D2 are determined by D0 and syndromes.");
println!("\nDetectors: {{S0=0, S1=0, D1^D0=0, D2^D0=0}}")

In [ ]:
// More complex example: 3-qubit circuit with interesting measurement dependencies
// 
// Circuit:
//   q0: --H--@-------M0
//            |
//   q1: -----X--H--@-M1
//                  |
//   q2: -----------X-M2
//
// This creates a chain where:
// - q0 and q1 become entangled (Bell pair)
// - Then q1 and q2 become entangled
// - Measuring in this order should show interesting dependencies

let mut sim = StdSymbolicSparseStab::new(3);

// Create the entanglement chain
sim.h(0);        // q0 in superposition
sim.cx(0, 1);    // Entangle q0-q1
sim.h(1);        // q1 in superposition (relative to q0)
sim.cx(1, 2);    // Entangle q1-q2

// Measure all qubits
let m0: SymbolicMeasurementResult = sim.mz(0);
let m1: SymbolicMeasurementResult = sim.mz(1);
let m2: SymbolicMeasurementResult = sim.mz(2);

println!("3-Qubit Chain Measurement Results:");
println!("  M0 (qubit 0): {:?} ^ {} (deterministic: {})", m0.outcome, m0.flip as u8, m0.is_deterministic);
println!("  M1 (qubit 1): {:?} ^ {} (deterministic: {})", m1.outcome, m1.flip as u8, m1.is_deterministic);
println!("  M2 (qubit 2): {:?} ^ {} (deterministic: {})", m2.outcome, m2.flip as u8, m2.is_deterministic);

println!("\nInterpretation:");
println!("  - M0 outcome is random and causes stab Z0 with sign: {:?}", m0.outcome);
println!("  - M1 outcome is random and causes stab Z1 with sign: {:?}", m1.outcome);
println!("  - M2 outcome is deterministic and depends on: {:?}", m2.outcome);

// Show what XOR means
if m2.outcome.len() == 2 {
    println!("\n  M2 = {{0, 1}} means: M2_outcome = M0_outcome XOR M1_outcome");
}

In [ ]:
// Trace through the 3-qubit circuit step by step with tableau display
// Note: The tableau format is now "{measurement_indices} ^ flip PauliString"
//   - {} ^ 0: identity sign, no flip (deterministic 0)
//   - {} ^ 1: identity sign, flipped (deterministic 1)
//   - {0} ^ 0: depends on measurement 0, no flip
//   - {0,1} ^ 1: XOR of measurements 0,1, flipped

let mut sim = StdSymbolicSparseStab::new(3);

println!("Initial state:");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(0);
println!("After H(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(0, 1);
println!("After CX(0,1):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(1);
println!("After H(1):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(1, 2);
println!("After CX(1,2):");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure
let m0: SymbolicMeasurementResult = sim.mz(0);
println!("After measuring qubit 0 (M0={:?} ^ {}, det={}):", m0.outcome, m0.flip as u8, m0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m1: SymbolicMeasurementResult = sim.mz(1);
println!("After measuring qubit 1 (M1={:?} ^ {}, det={}):", m1.outcome, m1.flip as u8, m1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m2: SymbolicMeasurementResult = sim.mz(2);
println!("After measuring qubit 2 (M2={:?} ^ {}, det={}):", m2.outcome, m2.flip as u8, m2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

In [ ]:
// Example where M2 depends on BOTH M0 and M1 (XOR)
//
// Circuit: Create a 3-qubit GHZ-like state, then do local rotations
//
//   q0: --H--@--------M0
//            |
//   q1: -----X--@-----M1
//               |
//   q2: --------X--H--M2
//
// The key insight: after CX gates, we have correlations.
// The H on q2 at the end converts Z2 to X2, which should
// create an XOR dependency when measured in Z basis.

let mut sim = StdSymbolicSparseStab::new(3);

println!("=== Circuit where M2 = M0 XOR M1 ===\n");

sim.h(0);
sim.cx(0, 1);
sim.cx(1, 2);
println!("After H(0), CX(0,1), CX(1,2) - GHZ state:");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(2);
println!("After H(2):");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure
let m0: SymbolicMeasurementResult = sim.mz(0);
println!("After M0 (outcome={:?} ^ {}, det={}):", m0.outcome, m0.flip as u8, m0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m1: SymbolicMeasurementResult = sim.mz(1);
println!("After M1 (outcome={:?} ^ {}, det={}):", m1.outcome, m1.flip as u8, m1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m2: SymbolicMeasurementResult = sim.mz(2);
println!("After M2 (outcome={:?} ^ {}, det={}):", m2.outcome, m2.flip as u8, m2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

println!("Summary:");
println!("  M0 = {:?} ^ {}", m0.outcome, m0.flip as u8);
println!("  M1 = {:?} ^ {}", m1.outcome, m1.flip as u8);
println!("  M2 = {:?} ^ {}", m2.outcome, m2.flip as u8);
if m2.outcome.len() == 2 && m2.outcome.contains(&0) && m2.outcome.contains(&1) {
    println!("\n  M2 = {{0,1}} means: M2_outcome = M0_outcome XOR M1_outcome");
}

In [ ]:
// Example demonstrating the flip flag from unitary operations
//
// The flip flag tracks phase changes from unitary gates:
// - X gate on |0⟩: Z stabilizer becomes -Z, so flip=1
// - Measurement of |1⟩ returns {} ^ 1 (deterministic 1)

let mut sim = StdSymbolicSparseStab::new(1);

println!("=== Demonstrating the flip flag ===\n");

println!("Initial state |0⟩:");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Apply X gate to flip |0⟩ to |1⟩
sim.x(0);
println!("After X gate (now |1⟩):");
println!("Stabilizers:\n{}", sim.stab_tableau());
println!("Notice: {{}} ^ 1 Z means the stabilizer is -Z (flip=1)\n");

// Measure - should be deterministic 1
let m: SymbolicMeasurementResult = sim.mz(0);
println!("Measurement result: {:?} ^ {} (det={})", m.outcome, m.flip as u8, m.is_deterministic);
println!("  - Empty outcome {{}} means no measurement dependencies");
println!("  - flip=1 means the result is flipped, giving outcome 1");

// Reset and show double-X cancellation
sim.reset();
sim.x(0);
sim.x(0);
println!("\nAfter X·X (identity):");
println!("Stabilizers:\n{}", sim.stab_tableau());
println!("Notice: {{}} ^ 0 Z means we're back to +Z (flip=0)")

In [ ]:
// Example combining measurement dependencies with flip flag
//
// Circuit: X on qubit 0, then create Bell state
//   q0: --X--H--@--M0
//               |
//   q1: --------X--M1
//
// This shows how the flip propagates through the circuit

let mut sim = StdSymbolicSparseStab::new(2);

println!("=== Combining measurement dependencies with flip ===\n");

sim.x(0);
println!("After X(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(0);
println!("After H(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(0, 1);
println!("After CX(0,1) - Bell state with phase:");
println!("Stabilizers:\n{}", sim.stab_tableau());

let m0: SymbolicMeasurementResult = sim.mz(0);
let m1: SymbolicMeasurementResult = sim.mz(1);

println!("Measurement results:");
println!("  M0: {:?} ^ {} (det={})", m0.outcome, m0.flip as u8, m0.is_deterministic);
println!("  M1: {:?} ^ {} (det={})", m1.outcome, m1.flip as u8, m1.is_deterministic);
println!("\nBoth measurements depend on measurement index 0.");
println!("The flips indicate phase accumulated from the initial X gate.")